# VLM-DENTAL — Chain-of-Thought Trace Generation & Verification

This notebook orchestrates autonomous CoT trace generation using **LangGraph**, a
multi-provider **Generator Pool** (primarily Gemini/NVIDIA NIM API, with local
`vLLM` as a supported-but-secondary option), and a multi-provider **Verifier
Pool** (NVIDIA NIM, Groq, OpenRouter, Gemini).

### Architecture Overview:
- **Cell 4c (Tool-Based vs No-Tools)**: choose which of the two trace kinds this
  run generates -- the main system's tool-based traces, or baseline #3's
  no-tools traces (dentex-agentic-vlm-proposal.md §6). See that cell's markdown
  for why these read/write separate files and why no-tools traces still need
  to be *generated*, not just prompted at eval time.
- **Cell 7a (Generate)**: Runs the configured generator provider, producing raw
  traces in `train_cot_traces_unverified.jsonl` (or `..._no_tools.jsonl`).
- **Cell 7b (Verify)**: Verifies pending traces against ground truth using
  external LLM verifiers with strict per-provider pacing and cooldown limits,
  writing passing traces to `train_cot_traces.jsonl` (or `..._no_tools.jsonl`).
- **Cell 7a/7b's `--git-sync-every`**: safely syncs traces to GitHub *during*
  generation/verification, via `dental_agent/training/git_sync.py` -- this is
  the one safe, multi-worker-aware push mechanism in this notebook (fetch +
  merge before push, relying on `.gitattributes`' `merge=union` rule, verified
  by hand to actually be necessary -- a plain merge conflicts even on two
  workers' genuinely disjoint appends). Prefer this over Cell 8 below.
- **Cell 8 (Manual Auto-Push)**: a manual, end-of-session catch-all that now
  reuses the SAME safe `git_sync.sync_and_push` -- not a second, different
  push mechanism. Mainly useful if you ran with `--git-sync-every 0` (synced
  push disabled) and want to push everything at the very end instead.
- Both generate and verify support incremental resume.


## 1. Mount Google Drive & Setup Workspace
Mounts Drive to access persistent storage and clones/pulls the latest `VLM-DENTAL` repository.

In [ ]:
import os

# ============================================================
#  TOGGLE: Set IS_COLAB = True for Google Colab, False for PC
# ============================================================
IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    work_dir = '/content/VLM-DENTAL'
    if not os.path.exists(work_dir):
        os.chdir('/content')
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
    os.chdir(work_dir)
    os.system('git pull')
else:
    # Local PC: ensure we are in the repo
    if not os.path.exists('VLM-DENTAL') and not os.path.exists('.git'):
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
        os.chdir('VLM-DENTAL')
    elif os.path.exists('VLM-DENTAL'):
        os.chdir('VLM-DENTAL')
    
    os.system('git pull')
    work_dir = os.getcwd()

print(f'Active working directory: {os.getcwd()}')
print(f'Mode: {"Google Colab" if IS_COLAB else "Local PC"}')


## 2. Install Dependencies
Installs `vLLM`, `langgraph`, `google-genai`, `ultralytics`, and repository dependencies.

> **Note on Runtime Restart:** If Colab prompts you with *"Restart runtime to use newly installed packages"*, click **Restart Session**, then skip this cell and continue directly from **Cell 3**.

In [ ]:
import sys
try:
    import vllm
    import langgraph
    import ultralytics
    print("✅ Dependencies already installed. Skipping pip install. (Fast boot!)")
except ImportError:
    print("⚠️ Missing dependencies detected. Running pip install...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[api]"])


## 3. Configure Credentials (.env as Single Source of Truth)
Automatically loads credentials in two layers:
1. **`.env` file (Primary)** — Auto-searches `/content/drive/MyDrive/VLM-DENTAL/.env` or the project root. If found, this is the **single source of truth** and Colab Secrets are ignored.
2. **Colab Secrets Tab (Fallback)** — If no `.env` is found, variables from the Colab Secrets tab (`google.colab.userdata`) are injected.

In [ ]:
import os, shutil, urllib.request
from dotenv import load_dotenv

# Layer 1: .env File
candidate_env_paths = [
    os.path.join(os.getcwd(), '.env'),
    '/content/drive/MyDrive/VLM-DENTAL/.env',
    '/content/drive/MyDrive/vlmdental/.env',
    '/content/drive/MyDrive/.env',
    '/content/VLM-DENTAL/.env',
    '/content/.env',
]

found_env = next((p for p in candidate_env_paths if os.path.exists(p) and os.path.getsize(p) > 0), None)
if found_env:
    local_env = os.path.join(os.getcwd(), '.env')
    if os.path.abspath(found_env) != os.path.abspath(local_env):
        shutil.copy(found_env, local_env)
        print(f'Copied .env from {found_env} to {local_env}')
    load_dotenv(local_env, override=True)
    print(f'Loaded credentials from single source of truth: {found_env}')
    print('Skipping Colab Secrets since .env exists.')
else:
    if not os.path.exists('.env') and os.path.exists('.env.example'):
        shutil.copy('.env.example', '.env')
        print('Created empty .env from .env.example template')
    print('No populated .env found — falling back to Colab Secrets tab...')

    # Layer 2: Overlay Colab Secrets (ONLY if .env was not found)
    try:
        from google.colab import userdata
        secret_keys = [
            'GEMINI_API_KEY', 'NVIDIA_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY',
            'NVIDIA_VERIFIER_MODEL', 'GROQ_VERIFIER_MODEL',
            'OPENROUTER_VERIFIER_MODEL', 'GEMINI_VERIFIER_MODEL',
            'GENERATOR_PROVIDER', 'GENERATOR_MODEL',
            'NVIDIA_GENERATOR_MODEL', 'GROQ_GENERATOR_MODEL',
            'OPENROUTER_GENERATOR_MODEL', 'GEMINI_GENERATOR_MODEL',
            'NVIDIA_COOLDOWN_SECONDS', 'NVIDIA_RPD_LIMIT',
            'GROQ_COOLDOWN_SECONDS', 'GROQ_RPD_LIMIT',
            'OPENROUTER_COOLDOWN_SECONDS', 'OPENROUTER_RPD_LIMIT',
            'GEMINI_COOLDOWN_SECONDS', 'GEMINI_RPD_LIMIT',
            'HF_TOKEN', 'GITHUB_TOKEN', 'DENTEX_IMAGES_REPO',
            'NVIDIA_VERIFIER_RPM_LIMIT', 'GROQ_VERIFIER_RPM_LIMIT',
            'OPENROUTER_VERIFIER_RPM_LIMIT', 'GEMINI_VERIFIER_RPM_LIMIT',
            'NVIDIA_GENERATOR_RPM_LIMIT', 'GROQ_GENERATOR_RPM_LIMIT',
            'OPENROUTER_GENERATOR_RPM_LIMIT', 'GEMINI_GENERATOR_RPM_LIMIT'
        ]
        overridden = []
        for key in secret_keys:
            try:
                val = userdata.get(key)
                if val:
                    os.environ[key] = val
                    overridden.append(key)
            except Exception:
                pass
        if overridden:
            print(f'Colab Secrets injected {len(overridden)} variable(s): {overridden}')
    except ImportError:
        pass

# Ensure sensible defaults for local trace generation
os.environ.setdefault('GENERATOR_PROVIDER', 'local')
os.environ.setdefault('GENERATOR_MODEL', 'QuantTrio/Qwen3.5-9B-AWQ')
os.environ.setdefault('LOCAL_VLLM_BASE_URL', 'http://localhost:8000/v1')

# Print status overview
print('\n' + '=' * 50)
print('ACTIVE CONFIGURATION OVERVIEW')
print('=' * 50)
print(f"GENERATOR_PROVIDER : {os.environ.get('GENERATOR_PROVIDER')}")
print(f"GENERATOR_MODEL    : {os.environ.get('GENERATOR_MODEL')}")
has_github = bool(os.environ.get('GITHUB_TOKEN', '').strip() and not os.environ.get('GITHUB_TOKEN', '').startswith('your_'))
print(f"GITHUB_TOKEN       : {'SET' if has_github else 'NOT SET'}")
active_verifiers = [
    p for p in ['NVIDIA', 'GROQ', 'OPENROUTER', 'GEMINI']
    if os.environ.get(f'{p}_API_KEY', '').strip() and not os.environ.get(f'{p}_API_KEY', '').startswith('your_')
]
print(f"Active Verifiers   : {active_verifiers if active_verifiers else 'NONE (add API keys in .env or Secrets tab)'}")
print('=' * 50)


## 4. Model Cache Directory Setup
Configures `HF_HOME` for model weights. In Colab mode, uses the fast local SSD (`/content/`). On a local PC, uses `data/models/vllm_cache/`.

In [ ]:
import os

# In Colab: use fast local SSD. On PC: use persistent project path.
if IS_COLAB:
    vllm_cache_dir = '/content/local_vllm_cache'
else:
    vllm_cache_dir = os.path.join('data', 'models', 'vllm_cache')

os.makedirs(vllm_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = vllm_cache_dir
os.environ['HF_HUB_CACHE'] = os.path.join(vllm_cache_dir, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(vllm_cache_dir, 'hub')

model_name = os.environ.get('GENERATOR_MODEL', 'QuantTrio/Qwen3.5-9B-AWQ')
hub_dir = os.path.join(vllm_cache_dir, 'hub')
model_slug = model_name.replace('/', '--')
cached_path = os.path.join(hub_dir, f'models--{model_slug}')

if os.path.exists(cached_path):
    cached_size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fns in os.walk(cached_path) for f in fns
    ) / (1024 * 1024)
    print(f'Model cached at {cached_path} ({cached_size_mb:.0f} MB)')
else:
    print(f'Model not cached yet. vLLM will download {model_name} to {vllm_cache_dir} on startup.')
print(f'Cache location: {vllm_cache_dir}')

## 4b. Choose Active Dataset
Which dataset this notebook run generates traces from.

- `dentex`: the primary dataset, all 4 diagnosis classes (Caries, Deep Caries, Periapical Lesion, Impacted).
- `tufts`: Tufts Dental Database (~202 images with Periapical Lesion findings). See `dental_agent/data/tufts.py`.

**Automatic File Naming:** `run_trace_gen.py` automatically isolates unverified traces per dataset (e.g. `train_cot_traces_unverified_dentex.jsonl` vs `train_cot_traces_unverified_tufts.jsonl`), so separate single-dataset runs never collide or skip records.


In [ ]:
# ============================================================
# Which dataset this run uses ("dentex" or "tufts").
# ============================================================
DATASET_NAME = "dentex"  # or "tufts"

# Optional explicit --output override (leave empty to use canonical dataset paths)
TRACE_OUTPUT_FLAG = ""

print(f"Active dataset: {DATASET_NAME}")
if TRACE_OUTPUT_FLAG:
    print(f"Output override: {TRACE_OUTPUT_FLAG}")


## 4c. Tool-Based vs No-Tools & Tufts All-Diseases Trace Generation
Which *kind* of trace this run generates -- independent of `DATASET_NAME` above.

- `NO_TOOLS = False` (default): the main system's traces -- full LangGraph tool loop, saved to `train_cot_traces_unverified_{dataset}.jsonl`.
- `NO_TOOLS = True`: baseline #3's SFT training data (no tools, single API turn), saved to `train_cot_traces_unverified_{dataset}_no_tools.jsonl`.
- `HEALTHY_ONLY = True`: negative control traces for clinician-verified normal scans.
- `TUFTS_ALL_DISEASES = True`: when `DATASET_NAME = "tufts"`, generates traces for all 4 native Tufts disease categories (Periapical, Non-Odontogenic, Pericoronal, Inter-Radicular) across ~280 abnormal images.


In [ ]:
# ============================================================
# Tool-based (main system) or no-tools (baseline #3 SFT data)?
# See the markdown cell above before setting this to True.
# ============================================================
NO_TOOLS = False
HEALTHY_ONLY = False  # Set True to generate negative control traces for clinician-verified normal scans
TUFTS_ALL_DISEASES = False  # Set True when DATASET_NAME="tufts" to generate traces across all 4 Tufts disease categories (~280 abnormal images)

# Automatically format extra CLI flags safely without shell backslash continuation hazards
GEN_EXTRA_FLAGS = " ".join(filter(None, [
    "--retry-failed" if RETRY_FAILED if "RETRY_FAILED" in globals() else False else None,
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

VERIFY_EXTRA_FLAGS = " ".join(filter(None, [
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

print(f"Mode: {'NO-TOOLS (baseline #3 SFT data)' if NO_TOOLS else 'tool-based (main system)'} | Healthy-Only: {HEALTHY_ONLY} | Tufts-All: {TUFTS_ALL_DISEASES}")


## 5. Download DENTEX Dataset (If Not Already Present)
Downloads and structures the DENTEX panoramic X-ray images needed for trace generation.

> [!NOTE]
> **First Run / Standalone Run:** Leave `DOWNLOAD_FULL_DATASET = True` to fetch the 15GB zip.
> 
> **Parallel / Targeted Run:** If you have already uploaded the dataset to your own HuggingFace repo (Step 11), you can set `DOWNLOAD_FULL_DATASET = False`. The generation script will use `DENTEX_IMAGES_REPO` from your `.env` to fetch **only the exact images required for your active slice** dynamically!


In [ ]:
# ============================================================
# TOGGLE: Set to False to skip the massive 15GB download if
# you are relying on dynamic slice downloading (DENTEX_IMAGES_REPO)
# ============================================================
DOWNLOAD_FULL_DATASET = False

if DOWNLOAD_FULL_DATASET:
    !python scripts/download_and_cleanup.py
else:
    print("Skipping full dataset download. (TraceGen will fetch images dynamically)")


## 5b. Locate Tufts Dataset (Alternative -- Access-Gated)
Unlike DENTEX, Tufts is **not** auto-downloadable -- it's gated behind a request form at
https://tdd.ece.tufts.edu/. Request access, download, and extract it yourself, then either:
- Set `TUFTS_LOCAL_DIR` in your `.env` to the extracted folder, or
- Drop it somewhere this notebook's working directory can find it (`find_local_tufts_dir`
  searches a few common locations for a folder name containing "tufts"/"TDD").

Only relevant if `DATASET_NAME = "tufts"` above. Skip this cell entirely for DENTEX runs.

In [ ]:
if DATASET_NAME == "tufts":
    from dental_agent.data.tufts import find_local_tufts_dir
    tufts_dir = find_local_tufts_dir()
    if tufts_dir:
        print(f"Found local Tufts dataset at: {tufts_dir}")
    else:
        print("No local Tufts dataset found. Request access at https://tdd.ece.tufts.edu/, "
              "extract it, and set TUFTS_LOCAL_DIR in .env to the extracted folder.")
else:
    print("DATASET_NAME is not 'tufts' -- nothing to do here.")

In [ ]:
# Delete unused partial sub-datasets to save disk space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

# Remove any lingering temporary download zip/cache folders
!rm -rf hf_cache
print('Unused dataset subsets and cache cleaned up successfully!')

In [ ]:
import os

# Ensure data/traces directory exists
os.makedirs('data/traces', exist_ok=True)

# Automatically fetch completed verified traces from Hugging Face Hub if missing locally
canonical_trace = 'data/traces/train_cot_traces.jsonl'
if not os.path.exists(canonical_trace) or os.path.getsize(canonical_trace) == 0:
    print('Completed traces not found locally. Syncing verified traces from Hugging Face (Reza-Nadimi/vlm-dental-traces)...')
    !python scripts/sync_traces_hf.py --download
else:
    size_mb = os.path.getsize(canonical_trace) / (1024 * 1024)
    print(f'Verified traces already present in data/traces/ ({size_mb:.1f} MB). Skipping download.')


## 6. Launch Local vLLM Generator Server
Spawns the local OpenAI-compatible vLLM inference server for `Qwen/Qwen3.5-9B` and polls `/v1/models` until it is ready.

> **Note:** Skip this cell if `GENERATOR_PROVIDER` is set to an external API instead of `local`.

In [ ]:
import subprocess
import time
import os
import urllib.request

# 1. Kill any existing zombie vLLM processes to prevent OOM errors!
print('Cleaning up any orphaned vLLM processes from previous runs...')
os.system('pkill -f "vllm.entrypoints.openai.api_server"')
time.sleep(2)  # Give GPU time to release memory

os.system('pip uninstall -y torchaudio > /dev/null 2>&1')
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

model_name = os.environ.get('GENERATOR_MODEL', 'QuantTrio/Qwen3.5-9B-AWQ')
download_dir = os.path.join(vllm_cache_dir, 'hub')
port = '8000'
log_path = 'vllm_server.log'

print(f'Starting vLLM server for {model_name}')
print(f'Model download dir: {download_dir}')

log_file = open(log_path, 'w', encoding='utf-8')

vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', model_name,
    '--port', port,
    '--max-model-len', '24576',
    '--download-dir', download_dir,
    '--trust-remote-code',
    '--enforce-eager',
    '--dtype', 'auto',
]

if IS_COLAB:
    # T4: use pre-quantized AWQ checkpoint via Marlin kernels.
    # --max-num-seqs matters here: this AWQ build only quantizes the standard
    # linear layers (vision encoder + GDN attention blocks stay higher-precision),
    # so weights alone take ~11.2GB of the T4's 15GB, leaving very little headroom.
    # vLLM's default max-num-seqs (256) sizes the startup KV-cache/encoder-cache
    # profiling pass for way more concurrency than that headroom supports, which
    # is why startup silently hangs instead of erroring. We only process one
    # radiograph at a time anyway, so capping this costs nothing.
    vllm_cmd += ['--quantization', 'awq_marlin',
                 '--gpu-memory-utilization', '0.95',
                 '--max-num-seqs', '1']
else:
    # Local PC with larger GPU: full precision
    vllm_cmd += ['--gpu-memory-utilization', '0.95',
                 '--max-num-seqs', '1']

print(f'Quantization: {"AWQ (Marlin kernels)" if IS_COLAB else "none (fp16)"}')
vllm_process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

base_url = f'http://localhost:{port}/v1'
max_wait = 900
poll_interval = 5
elapsed = 0

print('Starting vLLM server... (Streaming logs live)')
with open(log_path, 'r', encoding='utf-8', errors='ignore') as log_reader:
    while elapsed < max_wait:
        ret_code = vllm_process.poll()
        if ret_code is not None:
            print(f'\n❌ vLLM process exited with code {ret_code}.')
            raise RuntimeError(f'vLLM server terminated with return code {ret_code}')

        # Stream any new log lines directly to Colab stdout
        new_logs = log_reader.read()
        if new_logs:
            print(new_logs, end='', flush=True)

        try:
            req = urllib.request.Request(f'{base_url}/models')
            with urllib.request.urlopen(req, timeout=3) as resp:
                if resp.getcode() == 200:
                    print(f'\n>> ✅ vLLM server is READY! (took {elapsed}s)')
                    break
        except Exception:
            pass
        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        print(f'\nERROR: vLLM server did not become ready within {max_wait}s.')
        raise RuntimeError('vLLM server startup timeout')


## 7a. Generator Configuration (Choose ONE)
Run exactly **one** of the following cells to configure your trace generator. 
You can slice the dataset horizontally across Colab instances by adjusting `TOTAL_SLICES` and `SLICE_INDEX`.

In [ ]:
# --- Option 1: Local vLLM Generator ---
TOTAL_SLICES = 20
SLICE_INDEX = 1        # CHANGE THIS per Colab instance (1 to 20)
SLICE_SEED = 42
PACING_DELAY = 1.5      # Delay between successive images (seconds)
GIT_SYNC_EVERY = 5     # Checkpoint sync interval to git (0 = disabled)
MAX_TOOL_CALLS = 50  # Hard graph limit
MAX_TURNS = 25          # Ceiling on the per-image turn budget
MIN_TURNS = 5          # Floor on the per-image turn budget
TURNS_PER_FINDING_BUFFER = 5  # Budget = max(MIN_TURNS, n_findings + this), capped at MAX_TURNS
MAX_TOKENS = 4096       # Max tokens per generator turn (passed via CLI)
IMAGE_MAX_DIM = 0       # Image scaling max dimension (0 = unscaled full res)
PERTURB_SMALL_PROB = 0.25  # Probability of small synthetic bbox perturbation
PERTURB_BIG_PROB = 0.30    # Probability of big synthetic bbox perturbation
MAX_BLOBS_PER_TURN = 2     # Max action JSON blocks per turn before failing fast
MAX_PADDING_TURNS = 3      # Max consecutive mechanical padding turns
MAX_IDENTICAL_REPEATS = 3  # Max consecutive identical tool calls
RETRY_FAILED = True       # Auto-retry failed traces (3 immediate retries + 3 end-of-run retries)
MAX_RETRIES_PER_IMAGE = 3
MAX_SECOND_PASS_RETRIES = 3

GENERATOR_PROVIDER = "local"
GENERATOR_MODEL = "QuantTrio/Qwen3.5-9B-AWQ"


In [ ]:
# --- Option 2: Groq Generator ---
TOTAL_SLICES = 20
SLICE_INDEX = 1        # CHANGE THIS per Colab instance (1 to 20)
SLICE_SEED = 42
PACING_DELAY = 1.5      # Delay between successive images (seconds)
GIT_SYNC_EVERY = 5     # Checkpoint sync interval to git (0 = disabled)
MAX_TOOL_CALLS = 50  # Hard graph limit
MAX_TURNS = 25          # Ceiling on the per-image turn budget
MIN_TURNS = 5          # Floor on the per-image turn budget
TURNS_PER_FINDING_BUFFER = 5  # Budget = max(MIN_TURNS, n_findings + this), capped at MAX_TURNS
MAX_TOKENS = 2048       # Max tokens per generator turn (passed via CLI)
IMAGE_MAX_DIM = 1280    # Image scaling max dimension
PERTURB_SMALL_PROB = 0.25  # Probability of small synthetic bbox perturbation
PERTURB_BIG_PROB = 0.30    # Probability of big synthetic bbox perturbation
MAX_BLOBS_PER_TURN = 2     # Max action JSON blocks per turn before failing fast
MAX_PADDING_TURNS = 3      # Max consecutive mechanical padding turns
MAX_IDENTICAL_REPEATS = 3  # Max consecutive identical tool calls
RETRY_FAILED = True       # Auto-retry failed traces (3 immediate retries + 3 end-of-run retries)
MAX_RETRIES_PER_IMAGE = 3
MAX_SECOND_PASS_RETRIES = 3

GENERATOR_PROVIDER = "groq"
GENERATOR_MODEL = "qwen/qwen3.6-27b"


In [ ]:
# --- Option 3: NVIDIA NIM Generator ---
TOTAL_SLICES = 20
SLICE_INDEX = 1        # CHANGE THIS per Colab instance (1 to 20)
SLICE_SEED = 42
PACING_DELAY = 1.5      # Delay between successive images (seconds)
GIT_SYNC_EVERY = 5     # Checkpoint sync interval to git (0 = disabled)
MAX_TOOL_CALLS = 50  # Hard graph limit
MAX_TURNS = 25          # Ceiling on the per-image turn budget
MIN_TURNS = 5          # Floor on the per-image turn budget
TURNS_PER_FINDING_BUFFER = 5  # Budget = max(MIN_TURNS, n_findings + this), capped at MAX_TURNS
MAX_TOKENS = 16384      # Max tokens per generator turn (passed via CLI)
IMAGE_MAX_DIM = 0       # Image scaling max dimension (0 = unscaled full res)
PERTURB_SMALL_PROB = 0.25  # Probability of small synthetic bbox perturbation
PERTURB_BIG_PROB = 0.30    # Probability of big synthetic bbox perturbation
MAX_BLOBS_PER_TURN = 2     # Max action JSON blocks per turn before failing fast
MAX_PADDING_TURNS = 3      # Max consecutive mechanical padding turns
MAX_IDENTICAL_REPEATS = 3  # Max consecutive identical tool calls
RETRY_FAILED = True       # Auto-retry failed traces (3 immediate retries + 3 end-of-run retries)
MAX_RETRIES_PER_IMAGE = 3
MAX_SECOND_PASS_RETRIES = 3

GENERATOR_PROVIDER = "nvidia_nim"
GENERATOR_MODEL = "meta/muse-glimmer-30b"


In [ ]:
# --- Option 4: Gemini Generator ---
TOTAL_SLICES = 20
SLICE_INDEX = 1        # CHANGE THIS per Colab instance (1 to 20)
SLICE_SEED = 42
PACING_DELAY = 1.5      # Delay between successive images (seconds)
GIT_SYNC_EVERY = 5     # Checkpoint sync interval to git (0 = disabled)
MAX_TOOL_CALLS = 50  # Hard graph limit
MAX_TURNS = 25          # Ceiling on the per-image turn budget
MIN_TURNS = 5          # Floor on the per-image turn budget
TURNS_PER_FINDING_BUFFER = 5  # Budget = max(MIN_TURNS, n_findings + this), capped at MAX_TURNS
MAX_TOKENS = 16384      # Max tokens per generator turn (passed via CLI)
IMAGE_MAX_DIM = 0       # Image scaling max dimension (0 = unscaled full res)
PERTURB_SMALL_PROB = 0.25  # Probability of small synthetic bbox perturbation
PERTURB_BIG_PROB = 0.30    # Probability of big synthetic bbox perturbation
MAX_BLOBS_PER_TURN = 2     # Max action JSON blocks per turn before failing fast
MAX_PADDING_TURNS = 3      # Max consecutive mechanical padding turns
MAX_IDENTICAL_REPEATS = 3  # Max consecutive identical tool calls
RETRY_FAILED = True       # Auto-retry failed traces (3 immediate retries + 3 end-of-run retries)
MAX_RETRIES_PER_IMAGE = 3
MAX_SECOND_PASS_RETRIES = 3

GENERATOR_PROVIDER = "gemini"
GENERATOR_MODEL = "gemini-3.5-flash-lite"


In [ ]:
# --- Option 5: openrouter Generator ---
TOTAL_SLICES = 20
SLICE_INDEX = 1        # CHANGE THIS per Colab instance (1 to 20)
SLICE_SEED = 42
PACING_DELAY = 1.5      # Delay between successive images (seconds)
GIT_SYNC_EVERY = 5     # Checkpoint sync interval to git (0 = disabled)
MAX_TOOL_CALLS = 50  # Hard graph limit
MAX_TURNS = 25          # Ceiling on the per-image turn budget
MIN_TURNS = 5          # Floor on the per-image turn budget
TURNS_PER_FINDING_BUFFER = 5  # Budget = max(MIN_TURNS, n_findings + this), capped at MAX_TURNS
MAX_TOKENS = 16384      # Max tokens per generator turn (passed via CLI)
IMAGE_MAX_DIM = 0       # Image scaling max dimension (0 = unscaled full res)
PERTURB_SMALL_PROB = 0.25  # Probability of small synthetic bbox perturbation
PERTURB_BIG_PROB = 0.30    # Probability of big synthetic bbox perturbation
MAX_BLOBS_PER_TURN = 2     # Max action JSON blocks per turn before failing fast
MAX_PADDING_TURNS = 3      # Max consecutive mechanical padding turns
MAX_IDENTICAL_REPEATS = 3  # Max consecutive identical tool calls
RETRY_FAILED = True       # Auto-retry failed traces (3 immediate retries + 3 end-of-run retries)
MAX_RETRIES_PER_IMAGE = 3
MAX_SECOND_PASS_RETRIES = 3

GENERATOR_PROVIDER = "openrouter"
#GENERATOR_MODEL = "stealth/ox-alpha" #glm 5.3 flash
GENERATOR_MODEL = "minimax/minimax-m3:free"


## 7b. Run Trace Generation
Runs the LangGraph agent reasoning loop on DENTEX images, saving raw diagnostic traces into `data/traces/train_cot_traces_unverified.jsonl`.

- Uses local `vLLM` with **zero API rate limits**.
- Fully resumable: automatically skips already processed images.

In [ ]:
# Re-evaluate extra flags in case settings changed
gen_flags = " ".join(filter(None, [
    "--retry-failed" if RETRY_FAILED else None,
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

!python scripts/run_trace_gen.py --mode generate --split train \
    --dataset {DATASET_NAME} --total-slices {TOTAL_SLICES} --slice-index {SLICE_INDEX} --slice-seed {SLICE_SEED} \
    --pacing-delay {PACING_DELAY} --git-sync-every {GIT_SYNC_EVERY} \
    --max-tool-calls {MAX_TOOL_CALLS} --max-turns {MAX_TURNS} --min-turns {MIN_TURNS} --turns-per-finding-buffer {TURNS_PER_FINDING_BUFFER} \
    --max-tokens {MAX_TOKENS} --image-max-dim {IMAGE_MAX_DIM} \
    --perturb-small-prob {PERTURB_SMALL_PROB} --perturb-big-prob {PERTURB_BIG_PROB} \
    --max-blobs-per-turn {MAX_BLOBS_PER_TURN} --max-padding-turns {MAX_PADDING_TURNS} --max-identical-repeats {MAX_IDENTICAL_REPEATS} \
    --generator-provider {GENERATOR_PROVIDER} --generator-model {GENERATOR_MODEL} \
    --max-retries-per-image {MAX_RETRIES_PER_IMAGE} --max-second-pass-retries {MAX_SECOND_PASS_RETRIES} \
    --ignore-api-errors {gen_flags}


## 7c. Trace Health & Corruption Cleaner / Failed Trace Purge
Scans `train_cot_traces_unverified.jsonl` and purges historical multi-blob action dumps, unparsed XML artifacts (`<fake_tool_call>`), and `generation_failed` entries.
- Prevents duplicate `image_id` collisions when re-generating failed traces.
- Creates an automatic `.bak` backup before modifying.

In [ ]:
verify_flags = " ".join(filter(None, [
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

!python scripts/run_trace_gen.py --mode clean --dataset {DATASET_NAME} {verify_flags}


## 7d. Verifier Configuration (Choose ONE)
Configure your preferred verifier model for the verification and repair passes.
- **With-Tools Traces**: OpenRouter `minimax/minimax-m3:free` (Recommended)
- **No-Tools Traces**: Gemini `gemini-3.5-flash-lite` (Recommended)

In [ ]:
# --- Option 1 (Recommended for With-Tools Traces): OpenRouter MiniMax M3 ---
VERIFIER_PROVIDER = "openrouter"
VERIFIER_MODEL = "minimax/minimax-m3:free"
IMAGE_MAX_DIM = 0
MAX_REPAIRS = 1
GIT_SYNC_EVERY = 1
PACING_DELAY = 1.5


In [ ]:
# --- Option 2 (Recommended for No-Tools Traces): Google Gemini 3.5 Flash Lite ---
VERIFIER_PROVIDER = "gemini"
VERIFIER_MODEL = "gemini-3.5-flash-lite"
IMAGE_MAX_DIM = 0
MAX_REPAIRS = 1
GIT_SYNC_EVERY = 1
PACING_DELAY = 1.5


In [ ]:
# --- Option 3: NVIDIA NIM ---
VERIFIER_PROVIDER = "nvidia_nim"
VERIFIER_MODEL = "meta/muse-glimmer-30b"
IMAGE_MAX_DIM = 0
MAX_REPAIRS = 1
GIT_SYNC_EVERY = 1
PACING_DELAY = 1.5


In [ ]:
# --- Option 4: Groq ---
VERIFIER_PROVIDER = "groq"
VERIFIER_MODEL = "qwen/qwen3.6-27b"
IMAGE_MAX_DIM = 1280
MAX_REPAIRS = 1
GIT_SYNC_EVERY = 1
PACING_DELAY = 1.5


## 7e. Phase 1: Run Strict Verification Pass
Strictly validates diagnostic correctness against ground truth. Passing traces are promoted directly to `train_cot_traces.jsonl`.
- Nudge-aware: recognizes `nudge_crop` adjustments as intentional self-correction skills.
- Preserves multi-line clinical Chain-of-Thought reasoning.

In [ ]:
verify_flags = " ".join(filter(None, [
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

!python scripts/run_trace_gen.py --mode verify --split train \
    --dataset {DATASET_NAME} --total-slices {TOTAL_SLICES} --slice-index {SLICE_INDEX} --slice-seed {SLICE_SEED} \
    --pacing-delay {PACING_DELAY} --git-sync-every {GIT_SYNC_EVERY} \
    --image-max-dim {IMAGE_MAX_DIM} \
    --verifier-provider {VERIFIER_PROVIDER} --verifier-model {VERIFIER_MODEL} \
    --max-repairs {MAX_REPAIRS} \
    --ignore-api-errors {verify_flags}


## 7f. Phase 2: Run Verifier Self-Repair & Editor Pass
For candidate traces that were not verified in Phase 1, runs an intelligent LLM editing and repair pass using the verifier model + ground-truth clinical context.
- Fixes FDI quadrant/position errors, cleans pseudo-tool artifacts, and aligns with ground truth.
- Re-verifies and promotes passing repaired traces to `train_cot_traces.jsonl`.

In [ ]:
verify_flags = " ".join(filter(None, [
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

!python scripts/run_trace_gen.py --mode repair --split train \
    --dataset {DATASET_NAME} --total-slices {TOTAL_SLICES} --slice-index {SLICE_INDEX} --slice-seed {SLICE_SEED} \
    --pacing-delay {PACING_DELAY} --git-sync-every {GIT_SYNC_EVERY} \
    --image-max-dim {IMAGE_MAX_DIM} \
    --verifier-provider {VERIFIER_PROVIDER} --verifier-model {VERIFIER_MODEL} \
    --ignore-api-errors {verify_flags}


## 8. Manual Sync (End-of-Session Catch-All)
Uses the SAME safe sync mechanism as `--git-sync-every` above
(`dental_agent/training/git_sync.py`'s `sync_and_push`) -- fetch + merge
(relying on `.gitattributes`' `merge=union` rule) before push, not a second,
different push implementation. Mainly useful if you ran with
`--git-sync-every 0` and want to push everything at the very end, or just
want an explicit manual sync point. Safe to run even if Cells 7a/7b already
synced everything -- it's a no-op when there's nothing new.


In [ ]:
import os
import sys
sys.path.insert(0, '.')
from dental_agent.training.git_sync import sync_and_push

# Discovers all active trace files (tool-based + no-tools, unverified per dataset + verified combined)
candidate_paths = [
    'data/traces/train_cot_traces.jsonl',
    'data/traces/train_cot_traces_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified_dentex.jsonl',
    'data/traces/train_cot_traces_unverified_dentex_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified_tufts.jsonl',
    'data/traces/train_cot_traces_unverified_tufts_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified.jsonl',
    'data/traces/train_cot_traces_unverified_no_tools.jsonl',
]
existing_paths = [p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 0]

if not existing_paths:
    print('No traces to push yet. Run Trace Generation first.')
else:
    ok = sync_and_push(existing_paths, 'data: manual end-of-session sync from Colab')
    print('Sync succeeded.' if ok else 'Sync did not complete -- see [git-sync] messages above for why (often just nothing new to push, or a transient network issue -- safe to re-run).')


## 9. Status Dashboard & Pool Capacity
Inspects total generated vs verified traces and current API rate limit capacity.

In [ ]:
verify_flags = " ".join(filter(None, [
    "--no-tools" if NO_TOOLS else None,
    "--healthy-only" if HEALTHY_ONLY else None,
    "--tufts-all-diseases" if (TUFTS_ALL_DISEASES and DATASET_NAME == "tufts") else None,
    TRACE_OUTPUT_FLAG if TRACE_OUTPUT_FLAG else None,
]))

!python scripts/run_trace_gen.py --mode verify --split train \
    --dataset {DATASET_NAME} --total-slices {TOTAL_SLICES} --slice-index {SLICE_INDEX} --slice-seed {SLICE_SEED} \
    --verifier-provider {VERIFIER_PROVIDER} --verifier-model {VERIFIER_MODEL} \
    --status-only {verify_flags}


## 10. Manual Download Helper
Downloads verified trace files directly to your local computer via browser.

In [ ]:
from google.colab import files
import os

candidate_paths = [
    'data/traces/train_cot_traces.jsonl',
    'data/traces/train_cot_traces_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified_dentex.jsonl',
    'data/traces/train_cot_traces_unverified_dentex_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified_tufts.jsonl',
    'data/traces/train_cot_traces_unverified_tufts_no_tools.jsonl',
    'data/traces/train_cot_traces_unverified.jsonl',
    'data/traces/train_cot_traces_unverified_no_tools.jsonl',
]

downloaded = 0
for path in candidate_paths:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        files.download(path)
        print(f'Downloading {path}...')
        downloaded += 1

if downloaded == 0:
    print('No traces found to download.')


## 11. Upload Entire DENTEX Image Dataset to Hugging Face
Uploads the downloaded dataset images to a custom Hugging Face repo for faster future downloads.

**Prerequisite**: You must be logged into Hugging Face with a **WRITE** token. The cell below will use your `.env` token if present, or prompt you to login.

> [!NOTE]
> You only need to run this cell **ONCE** on your first Colab instance. It uploads the entire dataset. Once complete, you can safely launch all parallel Colabs, and they will dynamically download only what they need via `DENTEX_IMAGES_REPO`!


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
if hf_token and not hf_token.startswith('your_'):
    login(token=hf_token)
else:
    print('⚠️ HF_TOKEN not found in environment. Please login manually:')
    !huggingface-cli login

repo_id = os.environ.get('DENTEX_IMAGES_REPO')
if repo_id:
    print(f"\nUploading targeted subset to {repo_id}...")
    !python scripts/upload_dataset_images_to_hf.py --dataset dentex --repo-id {repo_id}
else:
    print("\nDENTEX_IMAGES_REPO not set. Skipping upload.")

## 11b. Upload Entire Tufts Image Dataset to Hugging Face (Now Available)
Same one-time, run-once-per-dataset upload pattern as DENTEX (Step 11) -- uploads Tufts
images plus a COCO-shaped annotation JSON built from its DataFrames to a lightweight
per-image HF repo, so parallel Colabs can later fetch only what they need via
`TUFTS_IMAGES_REPO`, the same way `DENTEX_IMAGES_REPO` already works.

`load_tufts_dataset`'s mapping is now implemented and verified -- this bundle covers the
~202 images with a tooth-mapped Periapical Lesion finding (the same scope trace-gen uses),
not the full 1,000-image corpus. YOLO training doesn't need this bundle at all -- it reads
`TUFTS_LOCAL_DIR` directly, covering all 1,000 images regardless of diagnosis.


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
if hf_token and not hf_token.startswith('your_'):
    login(token=hf_token)
else:
    print('⚠️ HF_TOKEN not found in environment. Please login manually:')
    !huggingface-cli login

repo_id = os.environ.get('TUFTS_IMAGES_REPO')
if repo_id:
    print(f"\nUploading Tufts dataset to {repo_id}...")
    !python scripts/upload_dataset_images_to_hf.py --dataset tufts --repo-id {repo_id}
else:
    print("\nTUFTS_IMAGES_REPO not set. Skipping upload.")

In [ ]:
!git pull

In [ ]:
!git fetch origin && git reset --hard origin/main
